In [ ]:
# Set knit directory
setwd(here::here('./05_euc_tereticornis'))
knitr::opts_knit$set(root.dir = setwd(here::here('./05_euc_tereticornis')))
getwd()

In [ ]:
options(scipen = 999)

# Save figures in specific place
knitr::opts_chunk$set(autodep        = TRUE,
                      cache          = FALSE,
                      cache.comments = TRUE,
                      comment = "",
                      # Save figures as pdf ?
                      #dev = c( "png", "pdf"),
                  
                      # Include code?
                      echo           = TRUE,
                      error          = FALSE,
                      fig.align      = "center",
                      
                      # Path where figures are going to be store pdf single 
                      # figures   
                      #fig.path       = paste0("./figures_euc_tereticornis", "/"),
                      fig.width      = 15,
                      fig.height     = 13,
                      message        = FALSE,
                      warning        = FALSE)

In [ ]:
library(HIEdroughtbox)
library(ggplot2)

# For data manipulation
library(dplyr)
library(purrr)
library(stringr)
library(tidyr)
library(here)

# For adding regression line to ggplot
library(ggpubr)

In [ ]:

tree_id_colors <- c("111" = "#eec000",
                   "112" = "#cf544c",
                   "113" = "#0175c3",
                   "114" = "#878687",
                   
                   "115" =  "#000000",
                   "116" =  "#008b28",
                   "117" =  "#8600b6",
                   "118" =  "#00a0ab")

string_colors <- c(string_1 = "#eec000",
                   string_2 = "#cf544c",
                   string_3 = "#0175c3",
                   string_4 = "#878687",
                   
                   string_5 =  "#000000",
                   string_6 =  "#008b28",
                   string_7 =  "#8600b6",
                   string_8 =  "#00a0ab")


# Function for adding the number of observations on top of a boxplot
give_n <- function(x){
  return(c(y = median(x)*2, label = length(x))) 
  # experiment with the multiplier to find the perfect position
}

# 1. Load data

## Load RWC data

In [ ]:
load("./euc_tereticornis_processed_data/euc_tereticornis_rwc_data.RData")

In [ ]:
# Show the names of each element in the list
glimpse(euc_tereticornis_rwc_data)

## Load Areas data file

In [ ]:
load("./euc_tereticornis_processed_data/euc_tereticornis_areas_data_cleaned.RData")

In [ ]:
euc_tereticornis_areas_data_for_water_loss <-
  
    euc_tereticornis_areas_data_cleaned %>%
  
        group_by(spcode, tree_id, set_temperature, string_number, vpd_control) %>% 
        
        # Convert cm2 to m2
        mutate(areas_m2 = areas_cm2/10000,
               double_sided_areas_m2 = double_sided_areas_cm2/10000,
               leaf_areas_m2 = if_else(organ == "leaves", areas_m2, 0)
               ) %>% 

        summarise(areas_m2 = sum(areas_m2),
                  
                  # Sum leaf areas by 0 to get just the leaf areas column
                  leaf_areas_m2 = sum(leaf_areas_m2),
                  
                  double_sided_areas_m2 = sum(double_sided_areas_m2))  %>% 
        
        mutate(set_temperature = as.integer(set_temperature), 
               .keep =  "unused") %>% 
        
        # Reorganize columns
        select(spcode, string_number, tree_id, set_temperature, everything()) %>% 
        arrange(vpd_control)

In [ ]:
glimpse(euc_tereticornis_areas_data_for_water_loss)

# 2. Plot climatic conditions inside the droughtbox

## VPD

In [ ]:

ggplot(data = euc_tereticornis_rwc_data, aes(x = factor(set_temperature),
                                      y = vpd_avg_kpa_avg,
                                      colour = vpd_control)) +
        geom_boxplot() +
        theme_bw(base_size = 14) +
  
        scale_y_continuous(breaks = scales::pretty_breaks(n = 12)) +
  
        ggtitle("VPD conditions inside the Droughtbox") +
        
        # Set colors
        scale_color_manual(values = c("#9e1523","#293352")) 

## Temperature

In [ ]:
ggplot(data = euc_tereticornis_rwc_data, aes(x = factor(set_temperature),
                                      y = tc_avg_deg_c_avg,
                                      colour = vpd_control)) +
    geom_boxplot() +
    theme_bw(base_size = 14) +
    scale_y_continuous(breaks = scales::pretty_breaks(n = 12)) +
    ggtitle("Temperature conditions inside the Droughtbox") +
    
    # Set colors
    scale_color_manual(values = c("#9e1523","#293352")) 

# 3. Calculate species water loss 

## Filter data in the 50% to 90% range RWC

In [ ]:
euc_tereticornis_rwc_data_filtered_data <-
    euc_tereticornis_rwc_data %>%
        filter(relative_water_content_perc <= 0.90 & relative_water_content_perc >= 0.50) 

### Plot filtered data

In [ ]:

euc_tereticornis_rwc_data_filtered_data %>%
  
  filter(vpd_control == "no") %>% 

  ggplot(data = ., aes(x = time_seconds, y = string_weight_grams,
                       colour = string_number)) +
      geom_point() + 
  
      # Increase the number of y-axis tick marks
      scale_y_continuous(breaks = scales::pretty_breaks(n = 5)) + 
  
      theme_bw(base_size = 14) +
  
      ggtitle("Data with no VPD control") +
  
      # Put legend at the bottom
      theme(legend.position = "none") +
  
      # Break plot into panels
      facet_grid(vars(set_temperature), vars(string_number)) +
  
      # Set colors  
      scale_color_manual(values = string_colors) +
      
      # Show slope and R2
      stat_cor(aes(label = ..rr.label..), size = 3, label.y = 38) +
      stat_regline_equation(label.y = 40,  size = 3)+
      scale_y_continuous(breaks = scales::pretty_breaks(n = 7)) +
      scale_x_continuous(breaks = scales::pretty_breaks(n = 7))

In [ ]:

euc_tereticornis_rwc_data_filtered_data %>% 
  
  filter(vpd_control == "yes") %>% 

  ggplot(data = ., aes(x = time_seconds,
                       y = string_weight_grams,
                       colour = string_number)) +
      geom_point() + 
  
      # Increase the number of y-axis tick marks
      scale_y_continuous(breaks = scales::pretty_breaks(n = 5)) + 
  
      theme_bw(base_size = 14) +
  
      ggtitle("Data with VPD below 2kPa") +
  
      # Put legend at the bottom
      theme(legend.position = "none") +
  
      # Break plot into panels
      facet_grid(vars(set_temperature), vars(string_number)) +
  
      # Set colors  
      scale_color_manual(values = string_colors) +
      
      # Show slope and R2
      stat_cor(aes(label = ..rr.label..), size = 3, label.y = 38) +
      stat_regline_equation(label.y = 40,  size = 3)+
      scale_y_continuous(breaks = scales::pretty_breaks(n = 7)) +
      scale_x_continuous(breaks = scales::pretty_breaks(n = 7))

### Filter data with poor R2 

In [ ]:
euc_tereticornis_data_for_water_loss <- 

    euc_tereticornis_rwc_data_filtered_data  

    # Remove data with poor R2
    # filter(!(vpd_control == "yes" & string_number == "string_8" & set_temperature == 55))
    
    # filter(!(vpd_control == "no" & string_number == "string_8" & set_temperature == 55)) 
  

In [ ]:
save(euc_tereticornis_data_for_water_loss, 
     file = "./euc_tereticornis_processed_data/euc_tereticornis_rwc_filtered_data.RData")

## Calculate the water loss rate of change   

In [ ]:
euc_tereticornis_slopes <-
    calculate_rate_of_change(euc_tereticornis_data_for_water_loss)

In [ ]:
# Make sure the maximum row length of the data frame slopes is:
# 8 (individuals)  * 6 (temperatures) + 8 (individuals)  * 7 (temperatures) = 48 + 56 = 104
nrow(euc_tereticornis_slopes)

In [ ]:
# Compare the slopes with those calculated in the ggplot
euc_tereticornis_slopes %>% 
    select(string_number, set_temperature, vpd_control, 
           slope_grams_per_second) %>% 
    
    reactable::reactable(.)

In [ ]:
ggplot(data = euc_tereticornis_slopes, aes(x = factor(set_temperature),
                                    y = slope_grams_per_second)) +
    geom_boxplot() +
    theme_bw(base_size = 14) +
    facet_wrap(~vpd_control) +
    stat_summary(fun.data = give_n, geom = "text", fun.y = median,
                 position = position_dodge(width = 0.75))

## Calculate transpiration rates

In [ ]:
euc_tereticornis_transpiration_rates <- 
  calculate_transpiration_rates(droughtbox_data = euc_tereticornis_data_for_water_loss,
                                leaf_and_branch_area_data = euc_tereticornis_areas_data_for_water_loss)

In [ ]:
ggplot(data = euc_tereticornis_transpiration_rates, aes(x = factor(set_temperature),
                                                 y = transpiration_single_grams_per_sec_m2)) +
    geom_boxplot() +
    theme_bw(base_size = 14) +
    facet_wrap(~vpd_control) +
    stat_summary(fun.data = give_n, geom = "text", fun.y = median,
                 position = position_dodge(width = 0.75))

In [ ]:
ggplot(data = euc_tereticornis_transpiration_rates, aes(x = factor(set_temperature),
                                                 y = transpiration_double_grams_per_sec_m2)) +
    geom_boxplot() +
    theme_bw(base_size = 14) +
    facet_wrap(~vpd_control) +
    stat_summary(fun.data = give_n, geom = "text", fun.y = median,
                  position = position_dodge(width = 0.75))

### Plot transpiration data

In [ ]:
ggplot(data = euc_tereticornis_transpiration_rates, aes(x = set_temperature,
                                                  y = transpiration_double_grams_per_sec_m2,
                             colour = factor(tree_id))) +
        
        theme_bw(base_size = 14) +
  
        geom_point(size = 3.5) +
  
        # Increase the tickmarks of the axes
        scale_y_continuous(breaks = scales::pretty_breaks(n = 12)) + 
        scale_x_continuous(breaks = scales::pretty_breaks(n = 7)) +
  
        # Position label at the bottom
        theme(legend.position = "bottom") +
    
        facet_wrap(~vpd_control) +
        
        # Set colors
        #scale_color_manual(values = c("#9e1523","#293352")) +
        ylab('Transpiration (grams*m-2*s-1)')+
        xlab("Temperature") +
        scale_color_manual(values = tree_id_colors) 

In [ ]:
# Add spcode to data before saving 

euc_tereticornis_water_loss_data <- 
  
  euc_tereticornis_transpiration_rates %>% 
    
  mutate(spcode = "euc_tereticornis") %>% 
  select(-c(median_vpd, median_vpd, median_rh, median_temp)) %>% 
  select(spcode, everything())

# 4. Export data

In [ ]:
save(euc_tereticornis_water_loss_data, 
     file = "./euc_tereticornis_processed_data/euc_tereticornis_water_loss_data.RData")